# Exploração Inicial - Ranking de Reclamações do BCB

Este notebook faz a primeira coleta e exploração do ranking de Reclamações publicado trimestralmente pelo DEATI

## Objetivos

1. Consumir a API oficial do RDR (Registro de Demanda do Cidadão)
2. Salvar o arquivo bruto em 'data/raw/' para cache local
3. Carregar com pandas e mapear estrutura, colunas e tipos
4. Identificar valores faltantes e inconsistentes

## Fonte 

Endpoint: '[https://www3.bcb.gov.br/rdrweb/rest/ext/ranking/arquivo](https://www3.bcb.gov.br/rdrweb/rest/ext/ranking/arquivo)'

Documentação Oficial: <https://dadosabertos.bcb.gov.br/dataset/ranking-de-instituicoes-por-indice-de-reclamacoes>


## Observações 

- o endpoint retorna CSV
- o ranking é publicado com defasagem aproximada de 60 dias após o trimestre 

In [1]:
import requests #HTTP
from pathlib import Path 
from datetime import datetime #timestamp para nomear os arquivos

#pastas de destino do cache local
DIR_RAW = Path("../data/raw") #../ pq o notebook esta em notebooks
DIR_RAW.mkdir(parents=True, exist_ok=True) 

print(f"Pasta de destino: {DIR_RAW.resolve()}")

Pasta de destino: C:\Users\Pedro Teixeira\observatorio-reclamacoes-bcb\data\raw


In [2]:
#Parametros de consulta - Ranking do 3 trimestre de 2025
params = {
    "ano": 2025,
    "periodicidade": "TRIMESTRAL",
    "periodo": 3,
    "tipo": "Bancos e financeiras",
}

URL_RANKING = "https://www3.bcb.gov.br/rdrweb/rest/ext/ranking/arquivo"

# GET com timeout - sempre bom definir timeout em chamadas externas
resposta = requests.get(URL_RANKING, params=params, timeout = 30)
resposta.raise_for_status() #lança erro se n for 2xx

print(f"Status: {resposta.status_code}")
print(f"URL final usadas: {resposta.url}")
print(f"tipo de conteúdo: {resposta.headers.get('Content-Type')}")
print(f"Tamanho da resposta: {len(resposta.content)} bytes")

#Ve os primeiros 500 bytes para detectar encoding e separador
amostra_bytes = resposta.content[:500]
try:
    amostra = amostra_bytes.decode("utf-8")
    encoding_detectado = "utf-8"
except UnicodeDecodeError:
    amostra = amostra_bytes.decode("latin-1")
    encoding_detectado = "latin-1"

print(f"\nEncoding provável: {encoding_detectado}")
print(f"\nPrimeiros 500 caracteres do arquivo:")
print(amostra)

Status: 200
URL final usadas: https://www3.bcb.gov.br/rdrweb/rest/ext/ranking/arquivo?ano=2025&periodicidade=TRIMESTRAL&periodo=3&tipo=Bancos+e+financeiras
tipo de conteúdo: application/octet-stream
Tamanho da resposta: 28358 bytes

Encoding provável: latin-1

Primeiros 500 caracteres do arquivo:
Ano;Trimestre;Categoria;Tipo;CNPJ IF;Instituição financeira;Índice;Quantidade total de reclamações respondidas;Quantidade de reclamações procedentes;Quantidade de reclamações procedentes extrapoladas;Quantidade total de reclamações analisadas;Quantidade total de clientes  CCS e SCR;Quantidade de clientes  CCS;Quantidade de clientes  SCR;
2025;3º;Top 15 - Bancos, Financeiras e Instituições de Pagamento;Conglomerado; ;99PAY IP (conglomerado);18,21;2493;136;453;749;24853840;24853838;2;
2025;3º;D


In [3]:
#Nome e timestamp para preservar o historico se a API mudar

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
nome_arquivo = f"ranking_2025_T3_bancos_{timestamp}.csv"
caminho_completo = DIR_RAW / nome_arquivo

#Escrita binaria wb
with open(caminho_completo, "wb") as f:
    f.write(resposta.content)

print(f"Arquivo salvo em: {caminho_completo}")
print(f"Tamanho em disco: {caminho_completo.stat().st_size} bytes")

Arquivo salvo em: ..\data\raw\ranking_2025_T3_bancos_20260507_174214.csv
Tamanho em disco: 28358 bytes


In [4]:
#CARREGAR COM PANDAS


import pandas as pd

#caminho do arquivo que acabei de salvar
arquivo_mais_recente = sorted(DIR_RAW.glob("ranking_2025_T3_bancos_*.csv"))[-1]
print(f"Carregando: {arquivo_mais_recente.name}")

#LEitura com os parametros descobertos na celula 3 (exploração)
df = pd.read_csv(
    arquivo_mais_recente,
    sep=";",
    encoding="latin-1",
    decimal=",",
    na_values=[" ", ""],
)

#Precisa pra remover a coluna fantasma que vai aparecer por causa do ; no final de cada linha
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

#Limpa os nomes nas colunas
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)

print(f"\nLinhas: {len(df)}")
print(f"\nColunas: {len(df.columns)}")
print(f"\nNomes das colunas:")
for c in df.columns:
    print(f"  - {c}")

Carregando: ranking_2025_T3_bancos_20260507_174214.csv

Linhas: 195

Colunas: 14

Nomes das colunas:
  - Ano
  - Trimestre
  - Categoria
  - Tipo
  - CNPJ IF
  - Instituição financeira
  - Índice
  - Quantidade total de reclamações respondidas
  - Quantidade de reclamações procedentes
  - Quantidade de reclamações procedentes extrapoladas
  - Quantidade total de reclamações analisadas
  - Quantidade total de clientes  CCS e SCR
  - Quantidade de clientes  CCS
  - Quantidade de clientes  SCR


In [5]:
# Estatísticas das colunas numéricas: count, mean, std, min, max, quartis
df.describe()

#Tipos de dados de cada coluna e memoria usada, procurar especialmente se indice virou float e quantidades viraram int
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 14 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   Ano                                                 195 non-null    int64  
 1   Trimestre                                           195 non-null    str    
 2   Categoria                                           195 non-null    str    
 3   Tipo                                                195 non-null    str    
 4   CNPJ IF                                             14 non-null     float64
 5   Instituição financeira                              195 non-null    str    
 6   Índice                                              69 non-null     str    
 7   Quantidade total de reclamações respondidas         195 non-null    int64  
 8   Quantidade de reclamações procedentes               195 non-null    int64  
 9   Quantidade

In [6]:
#  Padronização de nomes + conversão de tipos (IDEMPOTENTE 
import unicodedata

def normalizar_nome(nome: str) -> str:
    """Remove acentos e espaços, deixa em snake_case ASCII."""
    sem_acento = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("ascii")
    return sem_acento.strip().lower().replace(" ", "_")

# Padroniza nomes
df.columns = [normalizar_nome(c) for c in df.columns]

# Remove duplicatas de coluna
df = df.loc[:, ~df.columns.duplicated()]

# Converte indice so se ainda nao for numerico
# is_numeric_dtype pega int, float, etc... Negação pega object, str, string, etc
if not pd.api.types.is_numeric_dtype(df["indice"]):
    df["indice"] = (
        df["indice"]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    df["indice"] = pd.to_numeric(df["indice"], errors="coerce")
    print("Conversão de 'indice' aplicada.")
else:
    print(f"'indice' já está como {df['indice'].dtype}. Conversão pulada.")

# Converte cnpj_if so se ainda for numerico (porque depois de conversão vira string)
if pd.api.types.is_numeric_dtype(df["cnpj_if"]):
    df["cnpj_if"] = (
        df["cnpj_if"]
        .astype("Int64")
        .astype("string")
        .str.zfill(14)
    )
    print("Conversão de 'cnpj_if' aplicada.")
else:
    print(f"'cnpj_if' já está como {df['cnpj_if'].dtype}. Conversão pulada.")

# 5. Validação
print(f"\nTipo de 'indice': {df['indice'].dtype}")
print(f"Máximo: {df['indice'].max():.2f}")
print(f"Mínimo: {df['indice'].min():.2f}")
print(f"Primeiros não-nulos: {df['indice'].dropna().head().tolist()}")

Conversão de 'indice' aplicada.
Conversão de 'cnpj_if' aplicada.

Tipo de 'indice': float64
Máximo: 25308.57
Mínimo: 9.03
Primeiros não-nulos: [18.21, 614.12, 8305.63, 70.04, 1257.69]


In [7]:
from IPython.display import display

print("PRIMEIRAS 10 LINHAS")
display(df[["instituicao_financeira", "categoria", "indice",
"quantidade_de_reclamacoes_procedentes", "quantidade_total_de_clientes__ccs_e_scr"]].head(10))


print("\nESTATISTICAS DESCRITIVAS")
display(df[["indice", "quantidade_de_reclamacoes_procedentes", "quantidade_total_de_reclamacoes_analisadas", "quantidade_total_de_clientes__ccs_e_scr"]].describe())

PRIMEIRAS 10 LINHAS


,instituicao_financeira,categoria,indice,quantidade_de_reclamacoes_procedentes,quantidade_total_de_clientes__ccs_e_scr
0,99PAY IP (conglomerado),"Top 15 - Bancos, Financeiras e Instituições de...",18.21,136,24853840
1,ABC-BRASIL (conglomerado),"Demais Bancos, Financeiras e Instituições de P...",NaN,6,46241
2,ADYEN DO BRASIL IP (conglomerado),"Demais Bancos, Financeiras e Instituições de P...",NaN,2,129
3,AGIBANK (conglomerado),"Demais Bancos, Financeiras e Instituições de P...",614.12,990,6556525
4,AGORACRED S/A SCFI (conglomerado),"Demais Bancos, Financeiras e Instituições de P...",NaN,2,692346
5,AL5 S.A. SCFI (conglomerado),"Demais Bancos, Financeiras e Instituições de P...",NaN,4,20603
6,ALELO IP (conglomerado),"Demais Bancos, Financeiras e Instituições de P...",NaN,27,1697659
7,AME DIGITAL BRASIL IP (conglomerado),"Demais Bancos, Financeiras e Instituições de P...",8305.63,165,33864
8,ANDBANK (conglomerado),"Demais Bancos, Financeiras e Instituições de P...",NaN,3,112166
9,ASAAS GESTAO FINANCEIRA IP (conglomerado),"Demais Bancos, Financeiras e Instituições de P...",70.04,66,1149234



ESTATISTICAS DESCRITIVAS


,indice,quantidade_de_reclamacoes_procedentes,quantidade_total_de_reclamacoes_analisadas,quantidade_total_de_clientes__ccs_e_scr
count,69.000000,195.000000,195.000000,1.950000e+02
mean,699.423333,70.579487,276.353846,6.335080e+06
std,3194.098189,146.159945,520.635439,2.019986e+07
min,9.030000,0.000000,1.000000,0.000000e+00
25%,29.250000,1.000000,9.000000,1.866200e+04
50%,53.950000,7.000000,43.000000,3.057550e+05
75%,171.450000,63.000000,281.000000,2.366318e+06
max,25308.570000,990.000000,3437.000000,1.574757e+08


In [8]:
#TOP 15 do ranking - bancos, financeiros, IPs com mais de 4mi clientes

top_15 = (
    df[df["categoria"].str.contains("Top 15", na=False)]
    .sort_values("indice", ascending=False)
    .head(15)
    [["instituicao_financeira", "indice", "quantidade_de_reclamacoes_procedentes", "quantidade_total_de_clientes__ccs_e_scr"]]
    .reset_index(drop=True)
)

print("=" * 80)
print("Top 15 - Ranking de reclamações do BCB - Q3/2025")
print("=" * 80)
display(top_15)

Top 15 - Ranking de reclamações do BCB - Q3/2025


,instituicao_financeira,indice,quantidade_de_reclamacoes_procedentes,quantidade_total_de_clientes__ccs_e_scr
0,INTER (conglomerado),96.37,353,40066015
1,BANCO C6 (conglomerado),53.03,351,32544127
2,BRADESCO (conglomerado),51.74,474,110494968
3,MERCADO PAGO IP (conglomerado),51.58,512,66179957
4,PICPAY (conglomerado),50.06,702,65633305
5,PAGSEGURO (conglomerado),47.31,355,33715297
6,ITAU (conglomerado),45.13,520,100248541
7,NEON PAGAMENTOS IP (conglomerado),34.58,363,26111285
8,BTG PACTUAL/BANCO PAN (conglomerado),32.23,169,26578046
9,SANTANDER (conglomerado),29.25,176,69660783
